# 🍓 Strawberry Classifier

Complete YOLO11 workflow for strawberry detection and classification.

**Installation:** `pip install -e .`

## Setup

In [ ]:
from pathlib import Path

from strawberry_classifier import StrawberryClassifier, StrawberryDetector
from strawberry_classifier.data.crop_extractor import CropExtractor
from strawberry_classifier.data.prepare import prepare_dataset
from strawberry_classifier.data.utils import validate_dataset_structure

DATA_DIR = Path("data")
SOURCE_DATASET = "farm_dataset"

## Data Preparation

In [ ]:
source_dir = DATA_DIR / "raw" / SOURCE_DATASET
output_dir = DATA_DIR / "processed" / SOURCE_DATASET

if source_dir.exists():
    prepare_dataset(source_dir=source_dir, output_dir=output_dir, split_ratios=(0.7, 0.2, 0.1), seed=42)

    stats = validate_dataset_structure(output_dir)
    for split in ["train", "val", "test"]:
        images = stats.get(f"{split}_images", 0)
        labels = stats.get(f"{split}_labels", 0)
        print(f"{split}: {images} images, {labels} labels")
else:
    print(f"Source dataset not found: {source_dir}")

## Crop Extraction

In [ ]:
dataset_path = DATA_DIR / "processed" / SOURCE_DATASET
classification_output = DATA_DIR / "classification" / SOURCE_DATASET

if dataset_path.exists():
    extractor = CropExtractor(
        dataset_path=dataset_path,
        output_dir=classification_output,
        crop_size=(224, 224),
        padding=0.1,
        preserve_splits=True,
    )

    extractor.extract_crops()
    stats = extractor.get_stats()

    print(f"Total crops: {stats['total_crops']}")
    for class_name, count in stats["crops_per_class"].items():
        print(f"{class_name}: {count}")
else:
    print("Run data preparation first")

## Detection Training

In [ ]:
from pathlib import Path

dataset_path = Path(DATA_DIR) / "processed" / SOURCE_DATASET
data_yaml = dataset_path / "data.yaml"

if data_yaml.exists():
    detector = StrawberryDetector("yolo11n.pt")

    # Training with minimal epochs for testing
    results = detector.train(
        data=data_yaml,
        epochs=2,  # Very small for testing
        batch=4,  # Small batch size
        imgsz=640,
        project="runs/train",
        name="detection_test",
    )

    BEST_DETECTION_MODEL = f"{results.save_dir}/weights/best.pt"
    print(f"Model saved: {BEST_DETECTION_MODEL}")
else:
    print("Dataset not prepared")

## Classification Training

In [ ]:
if classification_output.exists():
    classifier = StrawberryClassifier("yolo11n-cls.pt")

    results = classifier.train(
        data=classification_output,
        epochs=50,
        batch=32,
        imgsz=224,
        project="runs/train",
        name="classification_experiment",
    )

    BEST_CLASSIFICATION_MODEL = f"{results.save_dir}/weights/best.pt"
    print(f"Model saved: {BEST_CLASSIFICATION_MODEL}")
else:
    print("Run crop extraction first")

## Validation

In [ ]:
if "BEST_DETECTION_MODEL" in locals():
    detector = StrawberryDetector(BEST_DETECTION_MODEL)
    results = detector.validate(data=data_yaml)

    if hasattr(results, "box"):
        print(f"mAP50: {results.box.map50:.4f}")
        print(f"mAP50-95: {results.box.map:.4f}")
else:
    print("No detection model to validate")

In [ ]:
if "BEST_CLASSIFICATION_MODEL" in locals():
    classifier = StrawberryClassifier(BEST_CLASSIFICATION_MODEL)
    results = classifier.validate(data=classification_output)

    if hasattr(results, "top1"):
        print(f"Top1 Accuracy: {results.top1:.4f}")
        print(f"Top5 Accuracy: {results.top5:.4f}")
else:
    print("No classification model to validate")

## Predictions

In [ ]:
image_path = "data/processed/farm_dataset/test/images/image_001.jpg"

if "BEST_DETECTION_MODEL" in locals():
    detector = StrawberryDetector(BEST_DETECTION_MODEL)
    results = detector.predict(image_path, save=True)
    print("Detection results saved")

if "BEST_CLASSIFICATION_MODEL" in locals():
    classifier = StrawberryClassifier(BEST_CLASSIFICATION_MODEL)
    results = classifier.predict(image_path, save=True)
    print("Classification results saved")

## Summary

Complete workflow for strawberry detection and classification using YOLO11.

## Dataset Utilities

Advanced dataset operations using integrated utilities.

In [ ]:
from strawberry_classifier.data import analyze_dataset, validate_dataset

# Validate dataset structure and format
dataset_path = "data/processed/farm_dataset"
validation_result = validate_dataset(dataset_path)

print(f"Dataset valid: {validation_result['valid']}")
print(f"Total issues: {validation_result['total_issues']}")

if validation_result["structure_issues"]:
    print("\nStructure Issues:")
    for issue in validation_result["structure_issues"]:
        print(f"  ❌ {issue}")

if validation_result["images_issues"]:
    print("\nImage/Label Issues:")
    for issue in validation_result["images_issues"][:5]:
        print(f"  ⚠️  {issue}")

stats = validation_result["stats"]
print("\nDataset Statistics:")
print(f"  Images: {stats['images']}")
print(f"  Labels: {stats['labels']}")
print(f"  Orphaned images: {stats['orphaned_images']}")
print(f"  Orphaned labels: {stats['orphaned_labels']}")

In [ ]:
# Analyze dataset label distribution
analysis = analyze_dataset(dataset_path)

print(f"Dataset Analysis: {dataset_path}")
print("=" * 50)

for split_name, counts in analysis["split_counts"].items():
    if not counts:
        continue

    print(f"\n{split_name.upper()} Split:")
    print("-" * 20)

    total_labels = sum(counts.values())
    print(f"Total labels: {total_labels}")

    for class_id in sorted(counts.keys()):
        class_name = (
            analysis["class_names"][class_id] if class_id < len(analysis["class_names"]) else f"Class_{class_id}"
        )
        count = counts[class_id]
        percentage = (count / total_labels) * 100
        print(f"  {class_name}: {count} ({percentage:.1f}%)")

In [ ]:
# Compare two datasets for overlap (example)
# dataset1_path = "data/processed/farm_dataset"
# dataset2_path = "data/processed/another_dataset"

# Uncomment to compare actual datasets:
# comparison_results = compare_datasets(dataset1_path, dataset2_path)
#
# print(f"\nDataset Comparison:")
# print("=" * 60)
#
# for split1, split_results in comparison_results.items():
#     print(f"\n{split1.upper()} split:")
#     print("-" * 30)
#
#     for split2, stats in split_results.items():
#         print(f"  vs {split2}: {stats['overlap_count']}/{stats['dataset1_count']} "
#               f"({stats['overlap_percentage']:.1f}%) images overlap")

print("Dataset comparison ready - uncomment code above to use with actual datasets")

### Dataset Operations

In [ ]:
# Merge two datasets (example)
# dataset1_path = "data/processed/farm_dataset"
# dataset2_path = "data/processed/another_dataset"
# output_path = "data/processed/merged_dataset"

# Uncomment to merge actual datasets:
# merge_stats = merge_datasets(
#     dataset1_path=dataset1_path,
#     dataset2_path=dataset2_path,
#     output_path=output_path,
#     dataset1_name="farm",
#     dataset2_name="additional"
# )
#
# print(f"Merge complete:")
# print(f"  Total images: {merge_stats['images']}")
# print(f"  Total labels: {merge_stats['labels']}")
# print(f"  Classes: {merge_stats['classes']}")

print("Dataset merging ready - uncomment code above to use")

In [ ]:
# Create dataset subset for quick testing
input_dataset = "data/processed/farm_dataset"
subset_output = "data/processed/farm_dataset_subset"

# Uncomment to create actual subset:
# subset_stats = create_subset(
#     dataset_path=input_dataset,
#     output_path=subset_output,
#     subset_ratio=0.1,  # 10% of original dataset
#     random_seed=42
# )
#
# print(f"Subset created:")
# print(f"  Original: {subset_stats['total']} images")
# print(f"  Subset: {subset_stats['selected']} images (10.0%)")

print("Subset creation ready - uncomment code above to create 10% subset")

### Color Transfer

In [ ]:
# Apply color transfer for data augmentation
source_dataset = "data/processed/farm_dataset"
reference_images = "data/reference_lighting"  # Directory with reference images
output_dataset = "data/processed/farm_dataset_color_transferred"

# Uncomment to apply color transfer:
# color_stats = apply_color_transfer_to_dataset(
#     dataset_path=source_dataset,
#     source_reference=reference_images,
#     output_path=output_dataset,
#     probability=0.5,  # Apply to 50% of images
#     intensity=1.0     # Full intensity
# )
#
# print(f"Color transfer complete:")
# print(f"  Processed: {color_stats['processed']} images")
# print(f"  Skipped: {color_stats['skipped']} images")

print("Color transfer ready - prepare reference images and uncomment code above")

### CLI Usage Examples

All dataset utilities are also available through the CLI:

In [ ]:
# CLI commands for dataset utilities:

print("Available CLI commands:")
print()
print("# Dataset validation:")
print("python -m strawberry_classifier validate-dataset --dataset data/processed/farm_dataset")
print()
print("# Dataset analysis:")
print("python -m strawberry_classifier analyze-dataset --dataset data/processed/farm_dataset")
print()
print("# Compare datasets:")
print("python -m strawberry_classifier compare-datasets --dataset1 data/dataset1 --dataset2 data/dataset2")
print()
print("# Merge datasets:")
print(
    "python -m strawberry_classifier merge-datasets --dataset1 data/dataset1 "
    "--dataset2 data/dataset2 --output data/merged"
)
print()
print("# Create subset:")
print("python -m strawberry_classifier create-subset --dataset data/input --output data/subset --ratio 0.1")
print()
print("# Color transfer:")
print(
    "python -m strawberry_classifier color-transfer --dataset data/input --reference reference_dir --output data/output"
)
print()
print("# View all available commands:")
print("python -m strawberry_classifier --help")

In [ ]:
print("Workflow completed:")
print(f"- Detection model: {'trained' if 'BEST_DETECTION_MODEL' in locals() else 'not trained'}")
print(f"- Classification model: {'trained' if 'BEST_CLASSIFICATION_MODEL' in locals() else 'not trained'}")
print("- Data prepared: True")
print("- Crops extracted: True")

## Oriented Bounding Box Visualization

The strawberry classifier includes tools for visualizing oriented bounding boxes (OBB) used in rotated object detection.

In [ ]:
from strawberry_classifier.visualization.plot_utils import plot_yolo_obb, visualize_dataset_obb_samples

### Visualize Dataset Samples

Visualize random samples from an oriented bounding box dataset to check data quality and annotation accuracy.

In [ ]:
# Visualize samples from OBB dataset (if available)
obb_dataset_path = DATA_DIR / "processed" / "farm_dataset_obb"

if obb_dataset_path.exists():
    print("Visualizing OBB dataset samples...")

    # Load class names if available
    classes_file = obb_dataset_path / "classes.txt"
    class_names = None
    if classes_file.exists():
        with open(classes_file, "r") as f:
            class_names = [line.strip() for line in f.readlines()]

    # Visualize samples from training set
    visualize_dataset_obb_samples(
        dataset_path=obb_dataset_path, num_samples=6, class_names=class_names, split="train", figsize=(15, 10)
    )
else:
    print(f"OBB dataset not found at: {obb_dataset_path}")
    print("Available datasets:")
    if (DATA_DIR / "processed").exists():
        for item in (DATA_DIR / "processed").iterdir():
            if item.is_dir():
                print(f"  - {item.name}")

### Visualize Individual Images

You can also visualize oriented bounding boxes on individual images.

In [ ]:
# Example: Visualize a specific image with its OBB annotations
if obb_dataset_path.exists():
    images_dir = obb_dataset_path / "train" / "images"
    labels_dir = obb_dataset_path / "train" / "labels"

    # Get first available image
    image_files = list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png"))

    if image_files:
        sample_image = image_files[0]
        sample_label = labels_dir / f"{sample_image.stem}.txt"

        print(f"Visualizing: {sample_image.name}")

        plot_yolo_obb(
            image_path=sample_image, label_path=sample_label, class_names=class_names, show_labels=True, figsize=(10, 8)
        )
    else:
        print("No image files found in the dataset")
else:
    print("OBB dataset not available for individual visualization")